# Level3 PPO Finetuning

This notebook warm-starts a new level3 PPO run from `level3_localobs_relax_final.ckpt`.

Default plan:
- start from the current best local-observation checkpoint;
- use a safer reward profile with more obstacle/crash pressure;
- train short runs first and evaluate every saved checkpoint on seeds 1-100.

Switch `TRAIN_ENV_TOML` to `level3_dr.toml` only when you want train-only DR finetuning.

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import wandb

from lsy_drone_racing.control.ppo_level3_observation import checkpoint_hidden_dim, unpack_checkpoint
from lsy_drone_racing.control.train_CleanRL_ppo_level3 import (
    Args,
    LEVEL3_OBSERVATION_LAYOUT,
    debug_rollout,
    train_ppo,
)

print("repo:", ROOT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("level3 observation layout:", LEVEL3_OBSERVATION_LAYOUT)

## Finetune Config

`level3.toml` is the recommended first pass because the current final checkpoint is still crash-heavy. After a safer nominal checkpoint improves, change `TRAIN_ENV_TOML` to `level3_dr.toml` and warm-start from that safer checkpoint.

In [ ]:
TRAIN_ENV_TOML = "level3.toml"
# TRAIN_ENV_TOML = "level3_dr.toml"  # enable this for train-only DR finetuning

RUN_NAME = "level3_localobs_safer_finetune_from_final"
INITIAL_MODEL_NAME = "checkpoints/level3_localobs_relax/level3_localobs_relax_final.ckpt"

CONFIG = TRAIN_ENV_TOML
CONFIG_PATH = ROOT / "config" / TRAIN_ENV_TOML
if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)

CONTROL_DIR = ROOT / "lsy_drone_racing/control"
INITIAL_MODEL_PATH = CONTROL_DIR / INITIAL_MODEL_NAME
if not INITIAL_MODEL_PATH.exists():
    raise FileNotFoundError(INITIAL_MODEL_PATH)

CHECKPOINT_DIR = CONTROL_DIR / "checkpoints" / RUN_NAME
MODEL_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_final.ckpt"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

JAX_DEVICE = "gpu"
CUDA = True

WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None
WANDB_MODE = "online"  # use "offline" if needed

NUM_ENVS_DEBUG = 4
NUM_ENVS_TRAIN = 1024
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 60_000_000
CHECKPOINT_INTERVAL_STEPS = 10_000_000

LEARNING_RATE = 5e-5
GAMMA = 0.99
GAE_LAMBDA = 0.95
UPDATE_EPOCHS = 5
NUM_MINIBATCHES = 8
ENT_COEF = 0.02
TARGET_KL = 0.03
HIDDEN_DIM = 256

print("config:", CONFIG_PATH)
print("initial:", INITIAL_MODEL_PATH)
print("output:", MODEL_PATH)

## Safer Reward Profile

Compared with `level3_localobs_relax`, this profile weakens the rush-through-gate incentive and increases safety pressure. The main target is to reduce obstacle/gate-frame crashes before adding stronger DR.

In [ ]:
REWARD_KWARGS = dict(
    progress_coef=0.0,
    gate_stage_coef=10.0,
    gate_axis_coef=12.0,
    near_gate_coef=0.0,

    gate_bonus=90.0,
    gate_back_bonus=12.0,
    finish_bonus=160.0,

    missed_gate_penalty=0.0,
    wrong_side_penalty=8.0,
    crash_penalty=100.0,

    obstacle_coef=8.0,
    obstacle_margin=0.40,
    obstacle_clearance_coef=6.0,

    timeout_penalty=80.0,
    time_penalty=0.03,

    act_coef=0.03,
    d_act_th_coef=0.10,
    d_act_xy_coef=0.10,
    cmd_tilt_coef=1.0,
    rpy_coef=1.0,
    tilt_limit_deg=40.0,
    tilt_excess_coef=15.0,
)

REWARD_KWARGS

In [ ]:
def build_args(**overrides):
    base = dict(
        config=CONFIG,
        total_timesteps=TOTAL_TIMESTEPS_TRAIN,
        num_envs=NUM_ENVS_TRAIN,
        num_steps=NUM_STEPS,
        learning_rate=LEARNING_RATE,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        update_epochs=UPDATE_EPOCHS,
        num_minibatches=NUM_MINIBATCHES,
        ent_coef=ENT_COEF,
        target_kl=TARGET_KL,
        hidden_dim=HIDDEN_DIM,
        cuda=CUDA,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        n_obs=2,
        **REWARD_KWARGS,
    )
    base.update(overrides)
    return Args.create(**base)

def torch_device(args):
    return torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

## Check Warm Start

This must print `obs_dim=68`, `hidden_dim=256`, and the same observation layout as the training wrapper.

In [ ]:
checkpoint = torch.load(INITIAL_MODEL_PATH, map_location="cpu")
model_state_dict, observation_layout = unpack_checkpoint(checkpoint)
initial_hidden_dim = checkpoint_hidden_dim(checkpoint, model_state_dict)
obs_dim = int(model_state_dict["actor_mean.0.weight"].shape[1])

print("observation_layout:", observation_layout)
print("obs_dim:", obs_dim)
print("hidden_dim:", initial_hidden_dim)

assert observation_layout == LEVEL3_OBSERVATION_LAYOUT
assert obs_dim == 68
assert initial_hidden_dim == HIDDEN_DIM

## W&B

Do not put API keys in the notebook. If needed, run `wandb login` in the terminal or let this cell prompt you.

In [ ]:
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ["WANDB_NOTEBOOK_NAME"] = str(ROOT / "notebooks/finetune_level3_ppo.ipynb")

if WANDB_ENABLED:
    if WANDB_MODE != "offline":
        wandb.login()
    print(f"W&B enabled: project={WANDB_PROJECT_NAME}, entity={WANDB_ENTITY}")
else:
    print("W&B disabled")

## Optional Debug Rollout

Run this before a long job if you changed config or reward coefficients. It uses zero actions and checks that the environment, reward wrapper, and observation shape compile.

In [ ]:
RUN_DEBUG_ROLLOUT = True

if RUN_DEBUG_ROLLOUT:
    debug_args = build_args(
        total_timesteps=NUM_ENVS_DEBUG * NUM_STEPS,
        num_envs=NUM_ENVS_DEBUG,
        debug_obs=True,
        debug_reward_every=1,
    )
    debug_rollout(debug_args, n_steps=20, device=torch_device(debug_args), jax_device=JAX_DEVICE)

## Train

This is a warm start, not a full optimizer resume. The policy/critic weights come from `INITIAL_MODEL_PATH`, while Adam starts fresh at `LEARNING_RATE`.

In [ ]:
args = build_args()
device = torch_device(args)

print("device:", device)
print("env:", args.config)
print("total_timesteps:", args.total_timesteps)
print("checkpoint_interval:", CHECKPOINT_INTERVAL_STEPS)
print("initial_model_path:", INITIAL_MODEL_PATH)
print("model_path:", MODEL_PATH)

train_ppo(
    args=args,
    model_path=MODEL_PATH,
    device=device,
    jax_device=JAX_DEVICE,
    wandb_enabled=WANDB_ENABLED,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_interval=CHECKPOINT_INTERVAL_STEPS,
    initial_model_path=INITIAL_MODEL_PATH,
)

## After Training

Evaluate each saved checkpoint with the same deterministic seed sweep used before: seeds `1-100`, `config/level3.toml`, and the inference controller path. Compare against the current baseline:

- `level3_localobs_relax_final`: success `15/100`, crash `85/100`, mean gates `1.38`, mean success time `6.40s`.

A checkpoint is worth keeping only if success increases without moving most failures from gate frames into obstacles.